# Test Notebooks to see if we can load some llms

In [1]:
%pip install -qU langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI

In [ ]:
inference_server_url = "https://user-jonasmorin-844854-vllm-user.lab.sspcloud.fr/v1/"

llm = ChatOpenAI(
    model="/root/.cache/huggingface/Phi-3.5-mini-instruct",
    openai_api_key="EMPTY",
    openai_api_base=inference_server_url,
    max_tokens=5,
    temperature=0,
)

In [ ]:
messages = [
    SystemMessage(
        content="You are a helpful assistant that translates English to Italian."
    ),
    HumanMessage(
        content="Translate the following sentence from English to Italian: I love programming."
    ),
]
llm.invoke(messages)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        (   "human", 
            "{input}"
        ),
    ]
)

chain = prompt | llm
chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

In [6]:
answer = chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

In [14]:
print(answer)
print(answer.content)
print(answer.response_metadata['token_usage']['total_tokens'])

content=' Ich liebe Programm' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 21, 'total_tokens': 26, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': '/root/.cache/huggingface/Phi-3.5-mini-instruct', 'system_fingerprint': None, 'finish_reason': 'length', 'logprobs': None} id='run-10f795dc-bad3-420d-a9e7-6f587a2ec45b-0' usage_metadata={'input_tokens': 21, 'output_tokens': 5, 'total_tokens': 26, 'input_token_details': {}, 'output_token_details': {}}
 Ich liebe Programm
26


In [19]:
from langchain_core.callbacks.base import BaseCallbackHandler
from typing import Dict, List, Any

class CustomHandler(BaseCallbackHandler):
    def on_llm_start(
        self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any
    ) -> Any:
        formatted_prompts = "\n".join(prompts)
        _log.info(f"Prompt:\n{formatted_prompts}")


output = chain.invoke({
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }, config={"callbacks": [CustomHandler()]})

Error in CustomHandler.on_llm_start callback: NameError("name '_log' is not defined")


In [22]:
prompt_as_string = prompt.format(
        input_language = "English",
        output_language =  "German",
        input =  "I love programming.",
)
print(prompt_as_string)

System: You are a helpful assistant that translates English to German.
Human: I love programming.


In [1]:
%pip install langchain
%pip install -U langchain-community
%pip install sentence-transformers
%pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 41.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 61.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 44.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 49.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 59.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 49.3 MB/s eta 0:00:0000:0100:01
   ━━━━━

In [5]:
from langchain import PromptTemplate, FewShotPromptTemplate

def few_shot_template_examples(example_prompt, example_selector):
    
  
    prefix = '''You are an helpful AI assistant who writes SQL query for a given question. Given the database described by the database schema below, write a SQL query that answers the question.\nDo not explain the SQL query.\nReturn just the query, so it can be run verbatim from your response.\n### Database Schema\n{table_info}
'''

    few_shot_prompt = FewShotPromptTemplate(
        # These are the examples we want to insert into the prompt.
        example_selector = example_selector,
        example_prompt = example_prompt,
        # The prefix is some text that goes before the examples in the prompt.
        # Usually, this consists of intructions.
        prefix= prefix,
        # The suffix is some text that goes after the examples in the prompt.
        # Usually, this is where the user input will go
        suffix= "### Question\n{input}\n### SQL query\n",
        # The input variables are the variables that the overall prompt expects.
        input_variables=["input", "table_info"],
        example_separator="\n\n",
    )
    return few_shot_prompt

In [24]:
import pandas as pd
from langchain import PromptTemplate, FewShotPromptTemplate
from langchain.prompts.example_selector import SemanticSimilarityExampleSelector
from langchain.vectorstores.chroma import Chroma
from langchain.embeddings import HuggingFaceEmbeddings


question = "test question"

table_name = "baby_names_favorite_firstname"

max_shot = 4

# def generate_sql_in_context_learning_similar_shots(question, table_name, n_shots = 2):
file_path = "../api/data/query_questions_db.csv"

# find the n_shots closest questions from the query_questions_db and the table
with open(file_path) as f:
    origin_of_shots = pd.read_csv(f, delimiter= ',')

#print(origin_of_shots)

examples = origin_of_shots.loc[origin_of_shots['db_id']==table_name]
examples = examples.reset_index()
few_shot_examples = []
meta_data = []

for j in range(len(examples)):
    ex_question = examples.loc[j,'question'].replace("\n","").strip()
    ex_query = examples.loc[j,'query']
    few_shot_examples.append({"question":ex_question})
    meta_data.append({"question":ex_question,"query":ex_query})

#print(examples)
#print(few_shot_examples)
#print(meta_data)

to_vectorize = [" ".join(example.values()) for example in few_shot_examples]

#print(to_vectorize)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")

if vectorstore is not None:
    ########CLEAR THE VECTORSTORE
    vectorstore.delete_collection()

vectorstore = Chroma.from_texts(to_vectorize, embeddings, metadatas=meta_data)

# Lower score is more similar
answers = vectorstore.similarity_search_with_score(query = question, k = max_shot)

#print(answers)

for item in answers:
    #print(item)
    print(item[0].metadata['question'])
    

# # print (f'######{question}###########')
# nl2sql_pairs=[]
# scores=[]
# for item in answers:
#     print(item[0].metadata['question']) # print out score          
#     # print(item[0].metadata['query']) 
#     # print(item[0].metadata['rules']) 
#     nl=item[0].metadata['question']
#     q=item[0].metadata['query']
#     score=item[1]
#     scores.append(score)
#     nl2sql_pairs.append(
#         {'question':nl, 'query':q,'score':score}
#     )
# # print('###########') 

examples_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k = max_shot,
)
examples_prompt = PromptTemplate(
    input_variables=["question","query"],
    template="### Question\n{question}\n### SQL query\n{query}",
)
prompt_template = few_shot_template_examples(examples_prompt, examples_selector)


All names that have been utilized by over 50 baby girls in canton Vaud. Please order the results by year.
How many baby names were registered?
How many favorite babynames are registered in year 2011?
Which year has the most registered baby names for all cantons?


In [25]:
print(prompt_template.pretty_repr())

You are an helpful AI assistant who writes SQL query for a given question. Given the database described by the database schema below, write a SQL query that answers the question.
Do not explain the SQL query.
Return just the query, so it can be run verbatim from your response.
### Database Schema
{table_info}


### Question
Show me all registered names in records.
### SQL query
SELECT DISTINCT bnff.first_name
FROM baby_names_favorite_firstname as bnff;


### Question
How many favorite babynames are registered in year 2011?
### SQL query
SELECT DISTINCT COUNT(bnff.first_name)
FROM baby_names_favorite_firstname as bnff
WHERE bnff.year = 2011;


### Question
How many baby names were registered?
### SQL query
SELECT DISTINCT COUNT(bnff.first_name)
FROM baby_names_favorite_firstname as bnff;


### Question
Which canton has most borned girls registed in 2011?
### SQL query
SELECT sum(bnff.amount), su.name
FROM baby_names_favorite_firstname as bnff
JOIN spatial_unit as su ON bnff.spatialunit_